# Part 2: Tokenizers

A model cannot read letters. A **tokenizer** cuts text into pieces called tokens and gives each piece an id. This part shows what it returns and why padding and the attention mask exist.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text = "Unbelievably dull, but the soundtrack was great!"
encoded = tokenizer(text)
print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

{'input_ids': [101, 4895, 8671, 2666, 3567, 6321, 10634, 1010, 2021, 1996, 6050, 2001, 2307, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'un', '##bel', '##ie', '##va', '##bly', 'dull', ',', 'but', 'the', 'soundtrack', 'was', 'great', '!', '[SEP]']


In [3]:
# Rare words are cut into pieces. ## marks a piece that continues a word.
print(tokenizer.tokenize("tokenization of unfathomable zeitgeist"))
print(tokenizer.vocab_size, tokenizer.model_max_length)

['token', '##ization', 'of', 'un', '##fat', '##hom', '##able', 'ze', '##it', '##ge', '##ist']
30522 512


## Padding and the attention mask

A batch is a rectangle, so shorter texts are padded with zeros. The attention mask tells the model which positions are real.

In [4]:
batch = tokenizer(["fine", "a much longer sentence here"], padding=True, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])

tensor([[ 101, 2986,  102,    0,    0,    0,    0],
        [ 101, 1037, 2172, 2936, 6251, 2182,  102]])
tensor([[1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]])


## Truncation

Models have a longest input. Anything longer must be cut, or it raises an error. You choose the limit.

In [5]:
long_text = "this sentence has far more words than the limit allows " * 3
ids = tokenizer(long_text, truncation=True, max_length=12)["input_ids"]
print(len(ids), tokenizer.decode(ids))

12 [CLS] this sentence has far more words than the limit allows [SEP]


## An uncased tokenizer lowercases everything

In [6]:
print(tokenizer("Great Movie")["input_ids"])
print(tokenizer("great movie")["input_ids"])

[101, 2307, 3185, 102]
[101, 2307, 3185, 102]


## Try it

1. Tokenize a sentence with a typo and see how the pieces change.
2. Tokenize a sentence in another language. What happens to the number of tokens?
3. Pad a batch of three texts to a fixed `max_length` of 20 and count the zeros in the mask.